# fast.ai Lesson 1–2 Redo — Intel Image Classification

**Added later** to fill a gap: the original coursework notebook for this assignment was never committed (see the folder README). Written now to satisfy the brief in `instructions.txt` — redo fast.ai Practical Deep Learning for Coders lessons 1 and 2 against a *different* problem than the course's own examples — but not executed in this environment (no `fastai`/`torch` installed here, no dataset download attempted). Every cell is standard `fastai` v2 API and should run as-is in Colab.

**Problem chosen:** scene classification on the [Intel Image Classification dataset](https://www.kaggle.com/datasets/puneet6060/intel-image-classification) (buildings / forest / glacier / mountain / sea / street) — a different task from the pets/digit classifiers used in the fast.ai lessons.


## Lesson 1 style — train a baseline classifier with transfer learning

In [ ]:
from fastai.vision.all import *

path = Path("intel-image-classification/seg_train/seg_train")

dls = ImageDataLoaders.from_folder(
    path,
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(224),
    batch_tfms=aug_transforms(size=224, min_scale=0.75),
)

dls.show_batch(max_n=9, figsize=(7, 7))


In [ ]:
learn = vision_learner(dls, resnet34, metrics=[error_rate, accuracy])
learn.fine_tune(4)


In [ ]:
learn.show_results(max_n=9, figsize=(7, 7))


## Lesson 2 style — interpret errors, clean data, fine-tune further

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6, 6))


In [ ]:
interp.plot_top_losses(9, nrows=3)


fast.ai's lesson 2 also covers using the `ImageClassifierCleaner` widget to weed out
mislabeled/bad images found via top losses, then retraining. That widget needs an
interactive Jupyter runtime (ipywidgets) to actually use — included here for
completeness, commented out since it can't run non-interactively.

In [ ]:
# from fastai.vision.widgets import ImageClassifierCleaner
# cleaner = ImageClassifierCleaner(learn)
# cleaner
#
# for idx in cleaner.delete():
#     cleaner.fns[idx].unlink()
# for idx, cat in cleaner.change():
#     shutil.move(str(cleaner.fns[idx]), path / cat)


In [ ]:
learn.unfreeze()
learn.lr_find()


In [ ]:
learn.fit_one_cycle(4, lr_max=slice(1e-6, 1e-4))


In [ ]:
learn.export("intel_scene_classifier.pkl")


## Lesson 2 style — inference on a new image

Loads the exported learner back and predicts on a single image, mirroring the
fast.ai lesson 2 "deploy your model" step.

In [ ]:
inf_learn = load_learner("intel_scene_classifier.pkl")
pred_class, pred_idx, probs = inf_learn.predict("sample_scene.jpg")
print(f"Predicted: {pred_class}  (confidence: {probs[pred_idx]:.3f})")
